# Agent Trust Demo

**The trust layer for multi-agent systems.**

An autonomous agent that verifies, scores, and controls other agents using cryptographic proof.

`pip install kanoniv-trust[hosted]`

In [ ]:
from agent_trust import TrustAgent

trust = TrustAgent(url="http://localhost:4100")
print(f"TrustAgent DID: {trust.did}")

## Register Agents with Capabilities
Each agent gets an Ed25519 key pair and a DID. Capabilities define what they CAN do.

In [ ]:
trust.register("researcher", capabilities=["search", "analyze", "fact-check"])
trust.register("writer", capabilities=["draft", "edit", "publish"])
trust.register("deployer", capabilities=["deploy", "rollback"])

## Delegate with Scopes, Caveats, and Expiry
Delegation controls what they're ALLOWED to do. Cryptographic, not advisory.

In [ ]:
trust.delegate("researcher", scopes=["search", "analyze"])
trust.delegate("writer", scopes=["draft", "edit"], caveats={"max_words": 1000})
trust.delegate("deployer", scopes=["deploy"], caveats={"env": "staging"}, expires_in=3600)

## Authorization Checks
Agents check before acting. The protocol enforces it.

In [ ]:
print(f"writer can draft:      {trust.authorized('writer', 'draft')}")
print(f"writer can publish:    {trust.authorized('writer', 'publish')}")
print(f"deployer can rollback: {trust.authorized('deployer', 'rollback')}")

## Scope Validation
Can't delegate what the agent can't do.

In [ ]:
try:
    trust.delegate("writer", scopes=["deploy"])
except Exception as e:
    print(f"Denied: {e}")

## Observe Outcomes
Every action is auto-signed with the agent's Ed25519 keys. Verified provenance, not logs.

In [ ]:
# Researcher does well
trust.observe("researcher", action="search", result="success", reward=0.9)
trust.observe("researcher", action="analyze", result="success", reward=0.85)
trust.observe("researcher", action="search", result="success", reward=0.8)

# Writer struggles
trust.observe("writer", action="draft", result="failure", reward=-0.3)
trust.observe("writer", action="edit", result="failure", reward=-0.5)
trust.observe("writer", action="draft", result="success", reward=0.4)

print("Outcomes recorded with signed provenance.")

## In-Context RL - The Moat
Agents read their own verified history before acting. No gradient descent - just structured memory from signed outcomes.

**Nobody else has this.**

In [ ]:
print("=== Researcher ===")
print(trust.recall("researcher").guidance)
print()
print("=== Writer ===")
print(trust.recall("writer").guidance)

## Reputation from Verified Provenance
Computed from signed outcomes. Not self-reported. Not LLM judgment.

In [ ]:
for name in ["researcher", "writer"]:
    rep = trust.reputation(name)
    sr = f"{rep.success_rate:.0%}" if rep.success_rate is not None else "N/A"
    ar = f"{rep.avg_reward:.2f}" if rep.avg_reward is not None else "N/A"
    print(f"{name}: score={rep.score}/100  success={sr}  reward={ar}  verified={rep.verified_actions}  scopes={rep.current_scopes}")

## Select Best Agent - UCB Exploration
Mathematically principled. Balances exploiting proven agents with exploring under-tested ones.

In [ ]:
best = trust.select(["researcher", "writer"], strategy="ucb")
print(f"Best agent: {best}")
print(f"Full ranking: {trust.rank(['researcher', 'writer', 'deployer'])}")

## Enforce - Restrict the Underperformer
Real authority, not a recommendation.

In [ ]:
trust.restrict("writer", scopes=["edit"])
print(f"writer can draft: {trust.authorized('writer', 'draft')}")
print(f"writer can edit:  {trust.authorized('writer', 'edit')}")

## Enforce - Revoke Entirely
Nuclear option. Agent can't act anymore.

In [ ]:
trust.revoke("writer")
print(f"writer can edit: {trust.authorized('writer', 'edit')}")
print(f"writer scopes:   {trust.reputation('writer').current_scopes}")

## Cryptographic Proof
Every agent has real Ed25519 keys. Signatures are verifiable.

In [ ]:
keys = trust.agent_keys("researcher")
print(f"DID: {keys.did}")
print(f"Signature: {keys.sign(b'proof of identity')}")

---

```
pip install kanoniv-trust
```

**GitHub:** [github.com/kanoniv/agent-trust](https://github.com/kanoniv/agent-trust)

MIT Licensed